# graphtope — iterative development notebook

**Carrier:** `topologicpy.Graph` (sole carrier). **Workflow:** develop here, extract stabilized
code into `graphtope/*.py`. **Scope:** M0 spike → M5 (reproduce the figure-5 Dom Narkomfin graph).

See `Topologic_Graph_Grammar_Spec.md` (spec), `CLAUDE.md` (carrier gotchas), and
`docs/Topologic_Carrier_Contribution_Briefing.md` (what to upstream to TopologicPy).


In [1]:
import topologicpy, networkx, graphtope
from graphtope import StateGraph, alphabet as A, serialize
print("topologicpy", topologicpy.__version__, "| networkx", networkx.__version__,
      "| graphtope", graphtope.__version__)

topologicpy 0.9.43 | networkx 3.6.1 | graphtope 0.1.0


## M0 — API spike (findings, already folded into the briefing note)

Measured against TopologicPy 0.9.43:

| Question | Result | Consequence |
|---|---|---|
| Parallel edges? | **deduped** (`Size==1`) | no multigraph at Stage 1; symmetry via `bidirectional` flag |
| Direction? | opt-in `directed=True`; `AdjacentVertices` ignores it | wrap traversal, always pass direction |
| Mutation? | **in-place**; Vertex identity **not** preserved | address by `id` via `VertexByKeyValue` |
| Edge dicts? | dropped unless `transferEdgeDictionaries=True`; bools→int; ontology keys injected | handled in `_topo.py` |
| Subgraph match? | vertices only, no edge/direction matcher | implement matcher in `rules.py` (M4) |

These are encoded as workarounds in `graphtope._topo` and `graphtope.model`.

## M1 — carrier + invariants

### Axiom A₀ (§7.0): two isolated generic blocks

In [2]:
G = StateGraph.axiom()
print(G)
print("nodes:", G.nodes())
print("b1 label/attrs:", G.node_label("b1"), G.node_attrs("b1"))
assert G.is_well_formed()

<StateGraph |N|=2 |E|=0 well_formed=True>
nodes: ['b1', 'b2']
b1 label/attrs: generic {'block': 'residential'}


### Build a small typed/directed graph by hand and validate (§2.2)

In [3]:
g = StateGraph()
g.add_node(A.GENERIC,   id="g1")
g.add_node(A.CORRIDOR,  id="c1", level=1)
g.add_node(A.STAIRCASE, id="s1")
g.add_edge("g1", "c1", A.H)            # horizontal adjacency (bidirectional)
g.add_edge("c1", "s1", A.V)            # vertical: c1 above s1 (one-way)

print(g)
print("edges:")
for e in g.edges():
    print("  ", e)
print("well-formedness errors:", g.well_formedness_errors())
print("H edge bidirectional:", g.edge("g1","c1")["bidirectional"],
      "| V edge bidirectional:", g.edge("c1","s1")["bidirectional"])
print("degrees  c1: out", g.out_degree("c1"), "in", g.in_degree("c1"))

<StateGraph |N|=3 |E|=2 well_formed=True>
edges:
   {'src': 'g1', 'tgt': 'c1', 'type': 'adjacency', 'orientation': 'H', 'bidirectional': True, 'weight': 1.0, 'attrs': {}}
   {'src': 'c1', 'tgt': 's1', 'type': 'adjacency', 'orientation': 'V', 'bidirectional': False, 'weight': 1.0, 'attrs': {}}
well-formedness errors: []
H edge bidirectional: True | V edge bidirectional: False
degrees  c1: out 1 in 1


### Invariants reject ill-formed inputs (§2.2)

In [4]:
for desc, thunk in [
    ("unknown label", lambda: g.add_node("not_a_label", id="x")),
    ("self-loop",     lambda: g.add_edge("g1", "g1", A.H)),
    ("bad orientation", lambda: g.add_edge("g1", "s1", "diagonal")),
    ("duplicate id",  lambda: g.add_node(id="g1")),
]:
    try:
        thunk(); print("NO ERROR (unexpected):", desc)
    except ValueError as ex:
        print("rejected:", desc, "->", ex)

rejected: unknown label -> unknown node label 'not_a_label'; Σ=['corridor', 'entrance', 'generic', 'l_section', 'staircase', 'u_section']
rejected: self-loop -> self-loops are not allowed (§2.2 inv-4)
rejected: bad orientation -> bad orientation 'diagonal'; Θ-orient=['H', 'V']
rejected: duplicate id -> duplicate node id 'g1'


### JSON round-trip (§10.1)

In [5]:
data = serialize.to_dict(g)
g2 = serialize.from_dict(data)
assert serialize.to_dict(g2) == data
import json; print(json.dumps(data, indent=2))

{
  "directed": true,
  "multigraph": true,
  "nodes": [
    {
      "id": "c1",
      "label": "corridor",
      "attrs": {
        "level": 1
      }
    },
    {
      "id": "g1",
      "label": "generic",
      "attrs": {}
    },
    {
      "id": "s1",
      "label": "staircase",
      "attrs": {}
    }
  ],
  "edges": [
    {
      "src": "c1",
      "tgt": "s1",
      "type": "adjacency",
      "orientation": "V",
      "bidirectional": false,
      "weight": 1.0,
      "attrs": {}
    },
    {
      "src": "g1",
      "tgt": "c1",
      "type": "adjacency",
      "orientation": "H",
      "bidirectional": true,
      "weight": 1.0,
      "attrs": {}
    }
  ]
}


### Visualise (topologic renderer; coordinates are layout only)

`G.show()` opens TopologicPy's Plotly viewer; `G.to_networkx()` is the escape-hatch view.

In [6]:
nxg = g.to_networkx()
print("networkx view:", nxg.number_of_nodes(), "nodes,", nxg.number_of_edges(), "edges")
# g.show()                 # uncomment for the interactive Plotly graph

networkx view: 3 nodes, 2 edges


## M2 — atomic basis A1–A7 (spec §4)

Seven reversible primitives. Each `op.apply(sg)` checks its precondition, performs
the effect, and **returns its inverse op** (capturing the pre-state the inverse needs).
So `inverse(op) ∘ op == id` — the reversibility at the heart of D-I / D-IV.

In [7]:
from graphtope.atomic import (AddNode, DelNode, AddEdge, DelEdge,
                               Relabel, Reweight, ReverseEdge)
from graphtope import serialize

g = StateGraph()
g.add_node(A.GENERIC, id="a"); g.add_node(A.GENERIC, id="b")
snapshot = serialize.to_dict(g)

op  = AddEdge("a", "b", A.V, weight=2.0)     # A3: a above b
inv = op.apply(g)                             # mutates g, returns the inverse (DelEdge)
print("after A3 :", g, "| inverse is", type(inv).__name__)
inv.apply(g)                                  # A4 undoes it
print("restored == snapshot:", serialize.to_dict(g) == snapshot)

after A3 : <StateGraph |N|=2 |E|=1 well_formed=True> | inverse is DelEdge
restored == snapshot: True


### Each atomic and its inverse, end to end

In [8]:
def reversible(build, op):
    g = build()
    before = serialize.to_dict(g)
    inv = op.apply(g)
    inv.apply(g)
    return serialize.to_dict(g) == before, type(op).__name__, "->", type(inv).__name__

def two_nodes():
    g = StateGraph(); g.add_node(A.GENERIC, id="a"); g.add_node(A.GENERIC, id="b"); return g
def two_nodes_edge():
    g = two_nodes(); g.add_edge("a", "b", A.H, weight=1.0); return g
def one_node():
    g = StateGraph(); g.add_node(A.STAIRCASE, id="s", level=2); return g

for ok, *what in [
    reversible(StateGraph, AddNode(A.CORRIDOR, {"level": 1})),  # A1 / A2
    reversible(one_node,   DelNode("s")),                       # A2 / A1
    reversible(two_nodes,  AddEdge("a", "b", A.V)),             # A3 / A4
    reversible(two_nodes_edge, DelEdge("a", "b")),              # A4 / A3
    reversible(one_node,   Relabel("s", A.CORRIDOR)),           # A5 / A5
    reversible(two_nodes_edge, Reweight("a", "b", 9.0)),        # A6 / A6
    reversible(two_nodes_edge, ReverseEdge("a", "b")),          # A7 / A7
]:
    print(f"{'OK ' if ok else 'FAIL'}  {what[0]:<11} inverse {what[2]}")

OK   AddNode     inverse DelNode
OK   DelNode     inverse AddNode
OK   AddEdge     inverse DelEdge
OK   DelEdge     inverse AddEdge
OK   Relabel     inverse Relabel
OK   Reweight    inverse Reweight
OK   ReverseEdge inverse ReverseEdge


### Property: `inverse(op) ∘ op == id` on random well-formed graphs

A quick in-notebook version of `tests/test_atomic.py::test_reversibility_property`.

In [9]:
import random
def random_graph(rng, n=6):
    g = StateGraph(); ids = [f"n{i}" for i in range(n)]
    for nid in ids:
        g.add_node(rng.choice(list(A.NODE_LABELS)), id=nid, level=rng.randint(0, 2))
    seen = set()
    for _ in range(n):
        a, b = rng.sample(ids, 2)
        if (a, b) in seen or (b, a) in seen: continue
        g.add_edge(a, b, rng.choice([A.H, A.V]), weight=round(rng.uniform(.5, 3), 2))
        seen.add((a, b))
    return g

rng = random.Random(0); fails = 0
for _ in range(200):
    g = random_graph(rng); before = serialize.to_dict(g)
    edges = [(e["src"], e["tgt"]) for e in g.edges()]
    if not edges: continue
    a, b = rng.choice(edges)
    inv = ReverseEdge(a, b).apply(g); inv.apply(g)
    fails += serialize.to_dict(g) != before
print("reverse-edge round-trips:", "ALL OK" if fails == 0 else f"{fails} FAILED")

reverse-edge round-trips: ALL OK


## M3 — core composite operations (spec §5)

The six generic verbs as recipes over the atomics. Each composite returns an
`OpSequence` (the reversed atomic inverses), so `inverse(op) ∘ op == id` exactly —
this is the "merge record / recovered π" of §5.1, for free.

In [10]:
from graphtope.composite import (Split, Merge, Divide, Union, Difference,
                                  Mirror, Transform, AttachPendant)

def ctx():
    g = StateGraph()
    g.add_node(A.GENERIC, id="m", area=10)
    g.add_node(A.CORRIDOR, id="L"); g.add_node(A.STAIRCASE, id="R")
    g.add_edge("L", "m", A.H, weight=2.0)
    g.add_edge("m", "R", A.H, weight=3.0)
    return g

g = ctx(); before = serialize.to_dict(g)
sp = Split("m", A.H)                       # divide m into two adjacent spaces
inv = sp.apply(g)
print("after SPLIT:", g, "children:", sp.children)
inv.apply(g)
print("inverse(SPLIT) ∘ SPLIT == id:", serialize.to_dict(g) == before)

after SPLIT: <StateGraph |N|=4 |E|=3 well_formed=True> children: ('n0', 'n1')
inverse(SPLIT) ∘ SPLIT == id: True


### SPLIT then MERGE returns to the start

In [11]:
g = ctx(); before = serialize.to_dict(g)
sp = Split("m", A.H); sp.apply(g)
c1, c2 = sp.children
Merge(c1, c2, result_id="m", result_label=A.GENERIC, result_attrs={"area": 10}).apply(g)
print("MERGE ∘ SPLIT == id (exact, with recovered id):", serialize.to_dict(g) == before)

MERGE ∘ SPLIT == id (exact, with recovered id): True


### MERGE coalesces a shared neighbour under ξ = max (and still inverts)

In [12]:
g = StateGraph()
g.add_node(A.GENERIC, id="a"); g.add_node(A.GENERIC, id="b"); g.add_node(A.CORRIDOR, id="x")
g.add_edge("a", "b", A.H); g.add_edge("a", "x", A.H, weight=2.0); g.add_edge("b", "x", A.H, weight=5.0)
before = serialize.to_dict(g)
inv = Merge("a", "b", result_id="ab").apply(g)
print("coalesced a/b → x weight (max 2,5):", g.edge("ab", "x")["weight"], "| edges:", g.size())
inv.apply(g)
print("un-coalesced exactly:", serialize.to_dict(g) == before)

coalesced a/b → x weight (max 2,5): 5.0 | edges: 1
un-coalesced exactly: True


### DIVIDE(k) = (k−1) chained SPLITs → a row of k cells (the apartment spine)

In [13]:
g = StateGraph(); g.add_node(A.GENERIC, id="blk")
d = Divide("blk", 4, A.H); d.apply(g)
print("cells:", d.children, "| nodes:", g.order(), "edges:", g.size())
for e in g.edges():
    print("   ", e["src"], "—", e["tgt"])

cells: ['n0', 'n2', 'n4', 'n5'] | nodes: 4 edges: 3
    n0 — n2
    n2 — n4
    n4 — n5


### MIRROR — reflect a wing across a seam (§5.3, the paper's open operation)

In [14]:
g = StateGraph()
g.add_node(A.CORRIDOR, id="c"); g.add_node(A.GENERIC, id="g1")
g.add_edge("c", "g1", A.H)
m = Mirror(["c", "g1"], seam=["c"]); m.apply(g)
print("after MIRROR:", g, "| copies:", m.copies)
print("seam stitch c — c':", g.has_edge("c", m.copies["c"]))

after MIRROR: <StateGraph |N|=4 |E|=3 well_formed=True> | copies: {'c': 'n0', 'g1': 'n1'}
seam stitch c — c': True


## Next: M4 — DPO rules + matcher (spec §6)

Production span `L ⊇ K ⊆ R`, typed-attributed **directed** subgraph monomorphism,
and NAC checking — the matcher topologic doesn't provide (briefing gap C3).